# 🤖 Web Scraping with Python — Notebook 2
## Selenium — Scraping JavaScript-Powered Websites

---

## 📌 Why Do We Need Selenium?

In Notebook 1, `requests + BS4` worked perfectly. But try scraping some modern websites and you'll hit a wall:

```python
response = requests.get('https://some-modern-site.com')
print(response.text)   # The data you want is NOT here!
```

**Why?** Because many websites load their content using **JavaScript AFTER the page loads**.
`requests` only gets the initial HTML — JavaScript never runs.

```
What requests sees:                What your browser sees:
─────────────────────────────────────────────────────────
<div id="products">                <div id="products">
  <!-- loaded by JS -->              <div class="card">iPhone 15</div>
</div>                               <div class="card">Samsung S24</div>
                                   </div>
    ↑ Empty! No data.                   ↑ Full of data!
```

### The Solution — Selenium

Selenium **controls a real browser** (Chrome, Firefox) through your Python code.
The browser runs JavaScript, loads everything — then you scrape the fully-loaded page.

```
requests + BS4    ->  Gets raw HTML only     (fast, no JS)
Selenium          ->  Controls real browser  (slower, full JS support)
```

### When to Use Which?

| Situation | Tool to Use |
|-----------|------------|
| Static HTML page (data visible in `response.text`) | `requests + BS4` ✅ |
| Page needs JS to load data | `Selenium` ✅ |
| Need to click buttons / fill forms | `Selenium` ✅ |
| Infinite scroll page | `Selenium` ✅ |
| Speed matters, no JS needed | `requests + BS4` ✅ |

> 💡 **Rule**: Always try `requests + BS4` first. Only switch to Selenium if the data is missing from `response.text`.

## ⚙️ Setup — Installing Selenium

### Option A: Manual Setup (Old way — tedious)
1. Install Selenium
2. Find your Chrome version (`chrome://version`)
3. Download matching ChromeDriver manually
4. Set the path in your code

### Option B: `webdriver-manager` (Recommended ✅ — automatic)
`webdriver-manager` automatically downloads the correct ChromeDriver for your Chrome version. No manual steps.

```bash
pip install selenium webdriver-manager
```

### What You Need
- ✅ **Google Chrome** installed on your computer
- ✅ **selenium** library
- ✅ **webdriver-manager** library (handles ChromeDriver automatically)

> 💡 **Check your Chrome version**: Open Chrome → type `chrome://version` in address bar → note the version number. `webdriver-manager` handles this for you automatically.

In [ ]:
# CELL 2 — Install and Import Everything

# Run this once to install (uncomment if not installed)
# !pip install selenium webdriver-manager

# ── Selenium core imports ──
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.keys import Keys

# ── webdriver-manager (auto ChromeDriver) ──
from webdriver_manager.chrome import ChromeDriverManager

# ── Standard imports ──
from bs4 import BeautifulSoup
import time
import json

print('All imports successful!')

## 🚀 Launching a Browser — Your First Selenium Script

Selenium opens a real Chrome window and controls it. Here's the minimal setup:

```python
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Launch Chrome
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

# Open a URL
driver.get('https://quotes.toscrape.com')

# Always close when done!
driver.quit()
```

### Key `driver` Properties

| Property / Method | What It Does |
|-------------------|--------------|
| `driver.get(url)` | Navigate to a URL |
| `driver.page_source` | Get the full HTML **after JS has run** |
| `driver.title` | Get the page title |
| `driver.current_url` | Get the current URL |
| `driver.quit()` | Close the browser completely |
| `driver.close()` | Close current tab only |

> ⚠️ **Always call `driver.quit()`** when done — otherwise Chrome processes keep running in the background and eat your RAM.

In [ ]:
# CELL 3 — Launch Browser and Fetch a Page

# ── Configure Chrome options ──
options = Options()
# options.add_argument('--headless')    # Uncomment to run WITHOUT opening a window
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-blink-features=AutomationControlled')  # anti-detection
options.add_experimental_option('excludeSwitches', ['enable-logging'])

# ── Launch Chrome (webdriver-manager auto-downloads the right ChromeDriver) ──
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# ── Navigate to a page ──
driver.get('https://quotes.toscrape.com')
time.sleep(2)    # Let the page load

# ── Basic info ──
print(f'Title       : {driver.title}')
print(f'Current URL : {driver.current_url}')
print(f'Page source length: {len(driver.page_source)} characters')
print()

# ── page_source -> pipe into BS4 for easy parsing ──
soup = BeautifulSoup(driver.page_source, 'html.parser')
first_quote = soup.find('span', class_='text')
print(f'First quote: {first_quote.text[:60] if first_quote else "Not found"}')

# ── Always quit! ──
driver.quit()
print('Browser closed.')

## 👻 Headless Mode — Run Without Opening a Window

In **headed mode** (default), Chrome opens a visible window. This is great for development — you can watch what your code is doing.

In **headless mode**, Chrome runs in the background with no visible window. Use this when:
- You're scraping on a server (no display available)
- You don't want windows popping up during a long scrape
- Running automated scripts

```python
options = Options()
options.add_argument('--headless')   # <- This one line makes it headless
```

### Recommendation

```
While learning / debugging  ->  Use headed (no --headless) — watch it work!
Final / production scraper  ->  Use headless — faster, no popup windows
```

### Helper Function — Create Driver
To avoid repeating setup code, we'll use this helper function throughout the notebook:

In [ ]:
# CELL 4 — Reusable Driver Setup Function

def get_driver(headless=False):
    """
    Creates and returns a configured Chrome WebDriver.
    headless=True  -> no browser window (faster)
    headless=False -> visible browser window (good for debugging)
    """
    opts = Options()

    if headless:
        opts.add_argument('--headless')

    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--disable-blink-features=AutomationControlled')
    opts.add_argument('--window-size=1920,1080')   # Set window size
    opts.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36'
    )
    opts.add_experimental_option('excludeSwitches', ['enable-logging'])

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opts
    )
    return driver


# ── Test the helper ──
driver = get_driver(headless=True)    # headless=True: no window opens
driver.get('https://quotes.toscrape.com')

print(f'Title   : {driver.title}')
print(f'URL     : {driver.current_url}')
print('Headless mode works!')

driver.quit()
# We'll use get_driver() in all examples from here on

## 🔍 Finding Elements — Selenium's Search Methods

Selenium has its own way to find elements (similar to BS4's `find()`):

| Method | Returns | When No Match |
|--------|---------|---------------|
| `driver.find_element(By.X, value)` | **First** matching element | Raises `NoSuchElementException` |
| `driver.find_elements(By.X, value)` | **List** of all matches | Returns empty list `[]` |

### Locator Strategies (`By.X`)

| Locator | Example | Use When |
|---------|---------|----------|
| `By.ID` | `By.ID, 'search-box'` | Element has unique `id` |
| `By.CLASS_NAME` | `By.CLASS_NAME, 'price'` | Element has a class |
| `By.CSS_SELECTOR` | `By.CSS_SELECTOR, 'p.price'` | Complex selection (recommended!) |
| `By.XPATH` | `By.XPATH, '//p[@class="price"]'` | When CSS doesn't work |
| `By.TAG_NAME` | `By.TAG_NAME, 'h2'` | Find by HTML tag |
| `By.NAME` | `By.NAME, 'username'` | Form input fields |
| `By.LINK_TEXT` | `By.LINK_TEXT, 'Next'` | Click exact link text |

> 💡 **Recommendation**: Use `By.CSS_SELECTOR` most of the time — same syntax you already know from BS4!

In [ ]:
# CELL 5 — Finding Elements with Different Locators

from selenium.common.exceptions import NoSuchElementException

driver = get_driver(headless=True)
driver.get('https://quotes.toscrape.com')
time.sleep(2)

# ── find_element -> first match ──
# By.CSS_SELECTOR (recommended)
first_quote = driver.find_element(By.CSS_SELECTOR, 'span.text')
print('First quote (CSS)  :', first_quote.text[:50])

# By.CLASS_NAME
first_author = driver.find_element(By.CLASS_NAME, 'author')
print('First author       :', first_author.text)

# By.TAG_NAME
first_h1 = driver.find_element(By.TAG_NAME, 'h1')
print('H1 text            :', first_h1.text)
print()

# ── find_elements -> all matches (list) ──
all_quotes = driver.find_elements(By.CSS_SELECTOR, 'div.quote')
print(f'Total quotes on page: {len(all_quotes)}')

# Loop through and print each
for i, q in enumerate(all_quotes[:3], 1):    # first 3 only
    text = q.find_element(By.CSS_SELECTOR, 'span.text').text
    print(f'  Quote {i}: {text[:50]}...')
print()

# ── Safe find (handle NoSuchElementException) ──
try:
    missing = driver.find_element(By.CSS_SELECTOR, 'div.does-not-exist')
except NoSuchElementException:
    print('Element not found - handled safely!')

driver.quit()

## 📦 What Can You Do With a Found Element?

Once you find an element, you can extract data from it or interact with it:

### Reading Data

| Property / Method | What It Returns | BS4 Equivalent |
|-------------------|-----------------|-----------------|
| `element.text` | Visible text content | `tag.text` |
| `element.get_attribute('href')` | Value of any HTML attribute | `tag.get('href')` |
| `element.get_attribute('innerHTML')` | Inner HTML as string | `str(tag)` |
| `element.tag_name` | HTML tag name | `tag.name` |
| `element.is_displayed()` | Is the element visible? | — |
| `element.is_enabled()` | Is the element enabled? | — |

### Interacting with Elements

| Method | What It Does |
|--------|--------------|
| `element.click()` | Click the element |
| `element.send_keys('text')` | Type text into input field |
| `element.clear()` | Clear an input field |
| `element.submit()` | Submit a form |

> 💡 **Tip**: You can also call `find_element()` ON an element (not just driver) to search within it — exactly like BS4's `tag.find()`.

In [ ]:
# CELL 6 — Element Properties and Attributes

driver = get_driver(headless=True)
driver.get('https://quotes.toscrape.com')
time.sleep(2)

# ── Reading element properties ──
quote_div = driver.find_element(By.CSS_SELECTOR, 'div.quote')

# .text -> visible text
print('Visible text (first 80 chars):')
print(quote_div.text[:80])
print()

# .tag_name -> HTML tag
print(f'Tag name: {quote_div.tag_name}')          # div
print(f'Is displayed: {quote_div.is_displayed()}') # True
print()

# .get_attribute() -> read any attribute
link = driver.find_element(By.CSS_SELECTOR, 'a')
print(f'Link href  : {link.get_attribute("href")}')
print(f'Link text  : {link.text}')
print()

# ── Searching WITHIN an element (like BS4's tag.find()) ──
all_quote_divs = driver.find_elements(By.CSS_SELECTOR, 'div.quote')

print('Extracting all quotes:')
print('-' * 50)
for div in all_quote_divs:
    # find_element on 'div' not 'driver' -> searches within this div only
    text   = div.find_element(By.CSS_SELECTOR, 'span.text').text
    author = div.find_element(By.CLASS_NAME, 'author').text
    tags   = [t.text for t in div.find_elements(By.CLASS_NAME, 'tag')]
    print(f'  Author : {author}')
    print(f'  Tags   : {tags}')
    print()

driver.quit()

## ⏳ Waits — The Most Important Selenium Concept

The biggest mistake beginners make with Selenium: **trying to find elements before they've loaded**.

JavaScript takes time to load. If you call `find_element()` too early, the element isn't in the page yet — you get a `NoSuchElementException`.

### 3 Types of Waits

**Type 1 — `time.sleep(n)` (Dumb wait)**
```python
driver.get(url)
time.sleep(3)              # Wait exactly 3 seconds, no matter what
element = driver.find_element(...)   # Then find
```
❌ Wasteful — waits even when page loaded in 0.5s

---

**Type 2 — Implicit Wait (Set once, applied globally)**
```python
driver.implicitly_wait(10)   # Wait UP TO 10s for any element, globally
```
✅ Better — but applies to every single `find_element()` call

---

**Type 3 — Explicit Wait (Recommended ✅ — wait for specific condition)**
```python
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, timeout=10)
element = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, 'div.quote'))
)   # Waits UP TO 10s, returns as soon as element appears
```
✅ Best — waits only as long as needed, for a specific element

| Wait Type | Use When |
|-----------|----------|
| `time.sleep()` | Quick testing / between page navigations |
| `implicitly_wait()` | Simple pages, global safety net |
| `WebDriverWait` | **Production scraping** — always prefer this |

In [ ]:
# CELL 7 — Waits in Action

from selenium.common.exceptions import TimeoutException

driver = get_driver(headless=True)
driver.get('https://quotes.toscrape.com')

# ── Method 1: time.sleep (simple but wasteful) ──
time.sleep(2)
quotes_check = driver.find_elements(By.CSS_SELECTOR, 'div.quote')
print(f'After sleep(2): {len(quotes_check)} quotes found')
print()

# ── Method 2: implicitly_wait (global fallback) ──
driver.implicitly_wait(10)    # Set once — applies to all find calls
h1 = driver.find_element(By.TAG_NAME, 'h1')
print(f'implicitly_wait found h1: {h1.text}')
print()

# ── Method 3: WebDriverWait + Expected Conditions (best) ──
wait = WebDriverWait(driver, timeout=10)

# Wait until quotes are present on the page
first_quote = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, 'div.quote'))
)
print(f'WebDriverWait found quote: {first_quote.text[:50]}...')
print()

# Wait for element to be clickable before clicking
next_btn = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'li.next a'))
)
print(f'Next button found and clickable: "{next_btn.text}"')

# ── Handling TimeoutException (when element never appears) ──
try:
    wait_short = WebDriverWait(driver, timeout=2)
    wait_short.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, 'div.does-not-exist'))
    )
except TimeoutException:
    print('Timed out waiting - element never appeared (handled safely!)')

driver.quit()

## 🖱️ Clicking Buttons & Filling Forms

This is where Selenium becomes truly powerful — it can **interact** with a page just like a human:

### The Click Pattern
```python
# Wait until the button is clickable (never click without waiting!)
wait = WebDriverWait(driver, 10)
button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'button.submit')))
button.click()
```

### The Form Fill Pattern
```python
# Find the input field
input_field = driver.find_element(By.NAME, 'username')

input_field.clear()                  # Clear any existing text first
input_field.send_keys('john_doe')    # Type your text

# Submit: either click the button or press Enter
input_field.send_keys(Keys.RETURN)   # Press Enter
# OR
driver.find_element(By.CSS_SELECTOR, 'input[type=submit]').click()
```

### Inspect First!
Before clicking or filling a form, **Inspect Element** the form:
- Find the `name` or `id` of each input field
- Find the CSS selector or text of the submit button
- Note what URL the form submits to

In [ ]:
# CELL 8 — Clicking Buttons and Filling Forms

from selenium.webdriver.common.keys import Keys

driver = get_driver(headless=False)   # headless=False so you can watch!
wait = WebDriverWait(driver, 10)

# ── Demo: Navigate using Next button ──
driver.get('https://quotes.toscrape.com')

print('Page 1 URL:', driver.current_url)
quotes_p1 = driver.find_elements(By.CSS_SELECTOR, 'div.quote')
print(f'Page 1 quotes: {len(quotes_p1)}')

# Click Next button
next_btn = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'li.next a'))
)
next_btn.click()        # Navigate to page 2!
time.sleep(2)

print('Page 2 URL:', driver.current_url)
quotes_p2 = driver.find_elements(By.CSS_SELECTOR, 'div.quote')
print(f'Page 2 quotes: {len(quotes_p2)}')
print()

# ── Demo: Fill the login form ──
driver.get('https://quotes.toscrape.com/login')
time.sleep(1)

print('Login page URL:', driver.current_url)

# Find username and password fields (inspect the page to know the name/id)
username = wait.until(EC.presence_of_element_located((By.ID, 'username')))
password = driver.find_element(By.ID, 'password')

username.clear()
username.send_keys('admin')       # Type username
print('Typed username: admin')

password.clear()
password.send_keys('admin')       # Type password
print('Typed password: admin')

# Submit the form
password.send_keys(Keys.RETURN)   # Press Enter to submit
time.sleep(2)

print('After login URL:', driver.current_url)
print('Login attempted! Check if redirected away from /login')

driver.quit()

## 📜 Scrolling — Handling Pages That Load on Scroll

Many modern sites (Twitter, Instagram, LinkedIn) load content **as you scroll down** — called **infinite scroll**.
`requests` gets absolutely nothing from these. Selenium can scroll just like a human.

### Scrolling Methods

```python
# Scroll to bottom of page
driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')

# Scroll by a fixed amount (pixels)
driver.execute_script('window.scrollBy(0, 500)')    # Scroll down 500px

# Scroll to top
driver.execute_script('window.scrollTo(0, 0)')

# Scroll an element into view
driver.execute_script('arguments[0].scrollIntoView()', element)
```

### Infinite Scroll Pattern

```python
last_height = driver.execute_script('return document.body.scrollHeight')

while True:
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')
    time.sleep(2)    # Wait for new content to load

    new_height = driver.execute_script('return document.body.scrollHeight')

    if new_height == last_height:    # No new content loaded -> we're at the end
        break
    last_height = new_height
```

In [ ]:
# CELL 9 — Scrolling

driver = get_driver(headless=True)
driver.get('https://quotes.toscrape.com')
time.sleep(2)

# ── Get initial page height ──
initial_height = driver.execute_script('return document.body.scrollHeight')
print(f'Initial page height : {initial_height}px')

# ── Scroll to bottom ──
driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')
time.sleep(1)

after_scroll = driver.execute_script('return document.body.scrollHeight')
print(f'After scroll height : {after_scroll}px')

# ── Scroll back to top ──
driver.execute_script('window.scrollTo(0, 0)')
time.sleep(1)
print('Scrolled back to top')
print()

# ── Infinite scroll simulation ──
# (quotes.toscrape.com is not infinite scroll, but the pattern is the same)
print('Simulating infinite scroll loop...')
last_height = driver.execute_script('return document.body.scrollHeight')
scroll_count = 0

for _ in range(3):    # Attempt 3 scrolls max (safety limit!)
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')
    time.sleep(2)

    new_height = driver.execute_script('return document.body.scrollHeight')
    scroll_count += 1
    print(f'  Scroll {scroll_count}: height = {new_height}px')

    if new_height == last_height:    # No new content -> stop
        print('  No new content loaded. Stopping.')
        break
    last_height = new_height

driver.quit()
print('Done!')

## 🤝 Selenium + BS4 — Best of Both Worlds

Selenium is powerful but **slower** for data extraction — it uses browser-level element finding.
BS4 is **fast and flexible** for parsing HTML.

The best pattern for production scrapers:

```
Selenium  ->  Handles JS, clicking, scrolling, navigation
BS4       ->  Does the actual data extraction from HTML
```

```python
# Step 1: Use Selenium to load the page (handles JS)
driver.get(url)
time.sleep(2)    # or WebDriverWait

# Step 2: Hand off page_source to BS4 for fast parsing
soup = BeautifulSoup(driver.page_source, 'html.parser')

# Step 3: Use BS4's familiar methods to extract data
items = soup.find_all('div', class_='product')
for item in items:
    name  = item.find('h2').text
    price = item.find('p', class_='price').text
```

### Why not use Selenium's `find_elements()` for everything?

| | Selenium `find_elements()` | BS4 `find_all()` |
|-|---------------------------|------------------|
| Speed | Slower (browser calls) | Faster (pure Python) |
| Syntax | More verbose | Cleaner |
| CSS Selectors | Supported | Supported |
| Works on static HTML | ✅ | ✅ |
| Works on JS pages | ✅ | ❌ (needs page_source) |

> Use Selenium for **navigation/interaction**, BS4 for **parsing**. Combine them!

In [ ]:
# CELL 10 — Selenium + BS4 Combined Pattern

driver = get_driver(headless=True)
wait   = WebDriverWait(driver, 10)

driver.get('https://quotes.toscrape.com')

# Step 1: Wait for page to load with Selenium
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div.quote')))

# Step 2: Hand off page_source to BS4
soup = BeautifulSoup(driver.page_source, 'html.parser')

# Step 3: Use BS4's familiar API to extract data
quote_boxes = soup.find_all('div', class_='quote')
print(f'Found {len(quote_boxes)} quotes using BS4 on Selenium page_source')
print()

all_quotes = []
for box in quote_boxes:
    text_tag   = box.find('span', class_='text')
    author_tag = box.find('small', class_='author')
    tag_links  = box.find_all('a', class_='tag')

    all_quotes.append({
        'quote' : text_tag.get_text(strip=True)   if text_tag   else 'N/A',
        'author': author_tag.get_text(strip=True) if author_tag else 'N/A',
        'tags'  : [t.get_text(strip=True) for t in tag_links]
    })

# Print results
for q in all_quotes[:3]:
    print(f'  {q["author"]:20s} | {q["quote"][:45]}...')

print()
print('Selenium loaded the page. BS4 parsed it. Best of both worlds!')

driver.quit()
print()
print('Batch 1 of 10 complete! Review and give green signal for Batch 2.')

## 📋 Handling Dropdowns — The `Select` Class

HTML `<select>` dropdowns can't be handled with a simple `.click()`. Selenium provides a special `Select` class for them.

```html
<select name="sort">
  <option value="price-asc">Price: Low to High</option>
  <option value="price-desc">Price: High to Low</option>
  <option value="rating">Rating</option>
</select>
```

### 3 Ways to Select an Option

```python
from selenium.webdriver.support.ui import Select

dropdown = Select(driver.find_element(By.NAME, 'sort'))

# Method 1: By visible text
dropdown.select_by_visible_text('Price: Low to High')

# Method 2: By value attribute
dropdown.select_by_value('price-desc')

# Method 3: By index (0 = first option)
dropdown.select_by_index(2)
```

### Useful Select Properties
```python
dropdown.options                # List of all options
dropdown.first_selected_option  # Currently selected option
dropdown.all_selected_options   # All selected (for multi-select)
```

In [ ]:
# CELL 11 — Handling Dropdowns with Select

from selenium.webdriver.support.ui import Select

# books.toscrape.com has a sort dropdown — perfect for practice!
driver = get_driver(headless=True)
wait   = WebDriverWait(driver, 10)

driver.get('https://books.toscrape.com')
time.sleep(2)

# Find the sort dropdown
dropdown_el = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, 'select.form-control'))
)
dropdown = Select(dropdown_el)

# See all available options
print('All dropdown options:')
for opt in dropdown.options:
    print(f'  value="{opt.get_attribute("value")}" | text="{opt.text}"')
print()

# Currently selected option
print(f'Currently selected: {dropdown.first_selected_option.text}')
print()

# Select by visible text
dropdown.select_by_visible_text('Price (low to high)')
time.sleep(2)    # Wait for page to reload with sorted results

print(f'After selecting: {dropdown.first_selected_option.text}')
print(f'URL after sort : {driver.current_url}')
print()

# Scrape first 3 book prices after sorting
soup = BeautifulSoup(driver.page_source, 'html.parser')
prices = [p.get_text(strip=True) for p in soup.find_all('p', class_='price_color')[:3]]
print('First 3 prices (sorted low to high):', prices)

driver.quit()

## 🔔 Alerts, Popups & iframes

### Alerts (JavaScript Popups)
JavaScript alerts are browser-level dialogs. Selenium can't click them with normal methods — use `switch_to.alert`:

```python
from selenium.webdriver.support import expected_conditions as EC

# Wait for alert to appear
wait.until(EC.alert_is_present())
alert = driver.switch_to.alert

print(alert.text)    # Read alert message
alert.accept()       # Click OK
# alert.dismiss()    # Click Cancel
# alert.send_keys('text')  # Type into prompt
```

### iframes (Embedded Pages)
An iframe is a webpage embedded inside another webpage. Selenium can't interact with iframe content directly — you must **switch into it first**:

```python
# Switch INTO the iframe
iframe = driver.find_element(By.TAG_NAME, 'iframe')
driver.switch_to.frame(iframe)       # Now inside the iframe

# Do your work inside the iframe
content = driver.find_element(By.CSS_SELECTOR, 'div.content')

# Switch BACK to main page
driver.switch_to.default_content()   # Must do this to return!
```

### Multiple Windows / Tabs
```python
original_window = driver.current_window_handle
driver.switch_to.window(driver.window_handles[-1])  # switch to new tab
driver.switch_to.window(original_window)             # switch back
```

In [ ]:
# CELL 12 — Alerts and iframes

from selenium.common.exceptions import NoAlertPresentException

driver = get_driver(headless=False)    # headed - alerts need a visible browser
wait   = WebDriverWait(driver, 10)

# ── Demo: Trigger and handle a JS alert ──
# We'll use a simple test page that has alerts
driver.get('https://the-internet.herokuapp.com/javascript_alerts')
time.sleep(2)

# Click 'Click for JS Alert' button
alert_btn = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[onclick="jsAlert()"]'))
)
alert_btn.click()

# Handle the alert
try:
    wait.until(EC.alert_is_present())
    alert = driver.switch_to.alert
    print(f'Alert message : "{alert.text}"')
    alert.accept()          # Click OK
    print('Alert accepted!')
except NoAlertPresentException:
    print('No alert appeared')

time.sleep(1)
print()

# ── Demo: iframes ──
driver.get('https://the-internet.herokuapp.com/iframe')
time.sleep(2)

# Find and switch into the iframe
iframe_el = wait.until(EC.presence_of_element_located((By.TAG_NAME, 'iframe')))
driver.switch_to.frame(iframe_el)
print('Switched INTO iframe')

# Now interact with content inside the iframe
try:
    body = driver.find_element(By.ID, 'tinymce')
    print(f'iframe content: "{body.text[:50]}"')
except Exception as e:
    print(f'Inside iframe: {e}')

# MUST switch back to main page before interacting with anything outside
driver.switch_to.default_content()
print('Switched back to main page')

driver.quit()

## 📸 Screenshots — Capture What Selenium Sees

Screenshots are **essential for debugging** — they show exactly what the browser saw at any moment. If your scraper fails, take a screenshot to understand why.

```python
# Screenshot of the entire page
driver.save_screenshot('page.png')

# Screenshot of a specific element only
element = driver.find_element(By.CSS_SELECTOR, 'div.card')
element.screenshot('element.png')

# Get screenshot as base64 (useful for embedding)
img_base64 = driver.get_screenshot_as_base64()
```

### When to Take Screenshots

```python
# In your except block — capture what went wrong!
try:
    element = wait.until(EC.presence_of_element_located(...))
except TimeoutException:
    driver.save_screenshot('error_screenshot.png')  # <- debug what you see
    print('Timed out! Check error_screenshot.png')
```

### Window Size
Some elements only appear at certain screen sizes. Always set window size:
```python
driver.set_window_size(1920, 1080)    # Full HD
driver.maximize_window()              # Maximize to screen size
```

In [ ]:
# CELL 13 — Screenshots

import os
os.makedirs('screenshots', exist_ok=True)

driver = get_driver(headless=True)
wait   = WebDriverWait(driver, 10)

driver.get('https://quotes.toscrape.com')
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div.quote')))

# ── Full page screenshot ──
driver.save_screenshot('screenshots/full_page.png')
print('Full page screenshot saved: screenshots/full_page.png')

# ── Element screenshot ──
first_quote_el = driver.find_element(By.CSS_SELECTOR, 'div.quote')
first_quote_el.screenshot('screenshots/first_quote.png')
print('Element screenshot saved : screenshots/first_quote.png')
print()

# ── Screenshot in error handling (best practice) ──
from selenium.common.exceptions import TimeoutException

try:
    # Try to find something that doesn't exist
    WebDriverWait(driver, 3).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, 'div.nonexistent'))
    )
except TimeoutException:
    driver.save_screenshot('screenshots/error_state.png')
    print('TimeoutException caught!')
    print('Error screenshot saved: screenshots/error_state.png')
    print('Open this file to see what the browser saw when it failed.')

print()
print('Screenshot files created:')
for f in os.listdir('screenshots'):
    size = os.path.getsize(f'screenshots/{f}')
    print(f'  {f} ({size:,} bytes)')

driver.quit()

## 🕵️ Anti-Detection — Don't Let Websites Know You're a Bot

Websites use several tricks to detect Selenium bots and block them. Here's how to stay under the radar.

### How Sites Detect Selenium

| Detection Method | What It Checks |
|-----------------|----------------|
| `navigator.webdriver` flag | JS property set to `true` by Selenium |
| User-Agent string | Contains 'HeadlessChrome' or 'Selenium' |
| Too-fast actions | Humans don't click in 0ms |
| Missing browser features | Plugins, languages, screen size |

### Fix 1 — Remove `webdriver` flag
```python
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option('excludeSwitches', ['enable-automation'])
options.add_experimental_option('useAutomationExtension', False)
```

### Fix 2 — Set a real User-Agent
```python
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0')
```

### Fix 3 — Human-like delays
```python
import random
time.sleep(random.uniform(1, 3))    # Random delay between actions
```

### Fix 4 — Set window size (headless gives unusual sizes)
```python
options.add_argument('--window-size=1920,1080')
```

> ⚠️ **Note**: Our `get_driver()` function already includes most of these fixes!

In [ ]:
# CELL 14 — Anti-Detection in Action

import random

# ── Check if Selenium is detectable ──
driver = get_driver(headless=True)
driver.get('https://quotes.toscrape.com')
time.sleep(2)

# Check navigator.webdriver value via JS
is_webdriver = driver.execute_script('return navigator.webdriver')
user_agent   = driver.execute_script('return navigator.userAgent')

print(f'navigator.webdriver : {is_webdriver}')     # None/False = good, True = detected
print(f'User-Agent          : {user_agent[:60]}...')
print()

# ── Human-like mouse movement using JavaScript ──
# Instantly jumping to elements looks robotic
# Use JS to scroll gradually instead

def human_scroll(driver, pixels=300):
    """Scroll gradually like a human, not all at once."""
    for i in range(0, pixels, 50):
        driver.execute_script(f'window.scrollBy(0, 50)')
        time.sleep(random.uniform(0.05, 0.15))   # tiny random pauses

print('Scrolling gradually (human-like)...')
human_scroll(driver, 400)
print('Done scrolling')
print()

# ── Random delay between actions ──
def random_delay(min_s=1, max_s=3):
    """Add a human-like random delay between scraping actions."""
    delay = random.uniform(min_s, max_s)
    time.sleep(delay)
    return delay

print('Using random delays:')
for i in range(3):
    d = random_delay(0.5, 1.5)
    print(f'  Action {i+1}: waited {d:.2f}s')

driver.quit()
print('Anti-detection demo complete!')

---
# 🧪 Mini Projects — Selenium in Action

---

## 🗣️ Mini Project 4 — JS Quotes Scraper with Selenium

**Goal**: Scrape all quotes from ALL pages of `quotes.toscrape.com/js/` — a JS-rendered version that `requests` cannot handle.

### Inspect First!
Open `https://quotes.toscrape.com/js/` → Right-click a quote → Inspect:

```
Same structure as before:
  <div class="quote">
    <span class="text">...
    <small class="author">...
    <a class="tag">...
  </div>

But this time, the data is INJECTED by JavaScript!
requests.get() -> empty page
Selenium       -> full data   (because browser runs JS)
```

### Strategy
```
1. Use Selenium to load each page (JS runs, data appears)
2. Wait for quotes to appear with WebDriverWait
3. Hand page_source to BS4 for fast parsing
4. Click Next button with Selenium to go to next page
5. Repeat until no Next button
6. Save all quotes to JSON
```

In [ ]:
# MINI PROJECT 4 — JS Quotes Scraper (Selenium + BS4 + Pagination)

driver = get_driver(headless=True)
wait   = WebDriverWait(driver, 10)

base_url  = 'https://quotes.toscrape.com/js/'
all_quotes = []
page_num   = 1

print('Starting JS quotes scrape...')
print('-' * 45)

driver.get(base_url)

while True:

    # Wait for quotes to load (JS must run first)
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div.quote')))
    except TimeoutException:
        print(f'Page {page_num}: Timed out waiting for quotes. Stopping.')
        break

    # Hand off to BS4 for fast parsing
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    boxes = soup.find_all('div', class_='quote')

    for box in boxes:
        text_tag   = box.find('span', class_='text')
        author_tag = box.find('small', class_='author')
        tag_links  = box.find_all('a', class_='tag')

        all_quotes.append({
            'quote' : text_tag.get_text(strip=True)   if text_tag   else 'N/A',
            'author': author_tag.get_text(strip=True) if author_tag else 'N/A',
            'tags'  : [t.get_text(strip=True) for t in tag_links],
            'page'  : page_num
        })

    print(f'Page {page_num:2d} | {len(boxes)} quotes | Total: {len(all_quotes)}')

    # Try to click Next button using Selenium
    try:
        next_btn = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'li.next a'))
        )
        next_btn.click()         # Navigate to next page
        page_num += 1
        time.sleep(random.uniform(1, 2))    # Polite delay

    except TimeoutException:
        print('No Next button — last page reached.')
        break

driver.quit()

print('-' * 45)
print(f'Total quotes scraped: {len(all_quotes)}')

# Save to JSON
with open('js_quotes.json', 'w', encoding='utf-8') as f:
    json.dump(all_quotes, f, indent=4, ensure_ascii=False)
print('Saved to js_quotes.json')

# Quick stats
from collections import Counter
top = Counter(q['author'] for q in all_quotes).most_common(3)
print('\nTop 3 authors:')
for author, count in top:
    print(f'  {author}: {count} quotes')

## 🔐 Mini Project 5 — Login + Scrape Protected Page

**Goal**: Login to `quotes.toscrape.com`, then scrape quotes from the page that appears only after login.

### Inspect the Login Form First!

Open `https://quotes.toscrape.com/login` → Inspect the form:

```html
<form method="POST" action="/login">
  <input type="text" id="username" name="username">
  <input type="password" id="password" name="password">
  <input type="submit" value="Login">
</form>
```

### Strategy
```
1. Go to /login page
2. Find username and password fields (by id)
3. Type credentials with send_keys()
4. Click submit button
5. Wait for redirect (confirms login success)
6. Scrape the page you now have access to
```

> 💡 **Important**: After login, the browser session (cookies) stays active.
> Selenium handles cookies automatically — just like a real browser staying logged in.

In [ ]:
# MINI PROJECT 5 — Login + Scrape Protected Content

driver = get_driver(headless=True)
wait   = WebDriverWait(driver, 10)

print('Step 1: Going to login page...')
driver.get('https://quotes.toscrape.com/login')
wait.until(EC.presence_of_element_located((By.ID, 'username')))
print(f'  URL: {driver.current_url}')
print()

# Step 2: Fill in credentials
print('Step 2: Filling login form...')
username_field = driver.find_element(By.ID, 'username')
password_field = driver.find_element(By.ID, 'password')

username_field.clear()
username_field.send_keys('admin')       # Any username works on this test site
time.sleep(random.uniform(0.3, 0.7))    # Human-like delay between fields

password_field.clear()
password_field.send_keys('admin')       # Any password works
print('  Credentials entered')
print()

# Step 3: Submit the form
print('Step 3: Submitting form...')
submit_btn = driver.find_element(By.CSS_SELECTOR, 'input[type="submit"]')
submit_btn.click()
time.sleep(2)    # Wait for redirect
print(f'  URL after login: {driver.current_url}')
print(f'  Login successful: {"/login" not in driver.current_url}')
print()

# Step 4: Scrape the logged-in page
print('Step 4: Scraping quotes as logged-in user...')
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div.quote')))

soup = BeautifulSoup(driver.page_source, 'html.parser')

# Check if logout link exists (confirms we are logged in)
logout = soup.find('a', href='/logout')
print(f'  Logout link present (confirms login): {logout is not None}')

# Scrape quotes
quotes = soup.find_all('div', class_='quote')
print(f'  Quotes found: {len(quotes)}')
print()

print('First 3 quotes after login:')
for q in quotes[:3]:
    text   = q.find('span', class_='text')
    author = q.find('small', class_='author')
    print(f'  {author.get_text(strip=True) if author else "?"}: {text.get_text(strip=True)[:45] if text else "?"}...')

driver.quit()
print('\nMini Project 5 complete!')

## ⚡ Selenium Quick Reference — Cheat Sheet

### Setup
```python
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument('--headless')           # Optional: no window
options.add_argument('--disable-blink-features=AutomationControlled')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
```

### Navigation
```python
driver.get(url)              # Open URL
driver.back()                # Browser back
driver.refresh()             # Reload page
driver.title                 # Page title
driver.current_url           # Current URL
driver.page_source           # Full HTML (feed to BS4!)
driver.quit()                # Close browser
```

### Finding Elements
```python
driver.find_element(By.CSS_SELECTOR, 'p.price')     # First match
driver.find_elements(By.CSS_SELECTOR, 'div.card')   # All matches -> list
element.find_element(By.TAG_NAME, 'a')              # Search within element
```

### Interacting
```python
element.click()                  # Click
element.send_keys('text')        # Type text
element.send_keys(Keys.RETURN)   # Press Enter
element.clear()                  # Clear input
element.text                     # Get visible text
element.get_attribute('href')    # Get attribute
```

### Waits (use these always!)
```python
wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div.quote')))
wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'button')))
wait.until(EC.alert_is_present())
```

### Special Interactions
```python
# Dropdown
Select(element).select_by_visible_text('Option 1')

# Alert
driver.switch_to.alert.accept()

# iframe
driver.switch_to.frame(iframe_element)
driver.switch_to.default_content()

# Scroll
driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')

# Screenshot
driver.save_screenshot('debug.png')
```

### Selenium + BS4 Pattern (use this always!)
```python
driver.get(url)
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'target')))
soup = BeautifulSoup(driver.page_source, 'html.parser')
data = soup.find_all('div', class_='item')    # Fast BS4 parsing
driver.quit()
```

---
# ✅ Notebook 2 — Complete!

## What You've Mastered

```
SELENIUM CORE
  [x] Why Selenium         When requests + BS4 fails (JS sites)
  [x] Setup                webdriver-manager auto ChromeDriver
  [x] get_driver()         Reusable helper with anti-detection
  [x] driver.get()         Navigate to any URL
  [x] page_source -> BS4   Best parsing combo
  [x] headless mode        Run without visible window

FINDING ELEMENTS
  [x] find_element()       First match (raises error if not found)
  [x] find_elements()      All matches -> list
  [x] By.CSS_SELECTOR      Recommended locator
  [x] By.ID / By.NAME      For form fields
  [x] element.text         Get visible text
  [x] get_attribute()      Get any HTML attribute

WAITS (most important concept!)
  [x] time.sleep()         Simple but wasteful
  [x] implicitly_wait()    Global fallback
  [x] WebDriverWait        Best - wait for specific condition
  [x] Expected Conditions  presence, clickable, alert_present

INTERACTIONS
  [x] click()              Click buttons, links
  [x] send_keys()          Type into inputs
  [x] Keys.RETURN          Press Enter
  [x] Select class         Handle dropdowns
  [x] switch_to.alert      Handle JS alerts
  [x] switch_to.frame      Work inside iframes
  [x] execute_script()     Run JS (scroll, etc.)

ANTI-DETECTION
  [x] Remove webdriver flag  --disable-blink-features
  [x] Fake User-Agent       Look like a real browser
  [x] Random delays         human_scroll(), random_delay()
  [x] Screenshots           Debug what Selenium sees

PROJECTS
  [x] Project 4  JS quotes scraper -> paginated -> JSON
  [x] Project 5  Login + scrape protected page
```

---

## 🎯 What's in Notebook 3

Now that you know both tools deeply, Notebook 3 combines them in **real production-style patterns**:

| Topic | What You'll Build |
|-------|------------------|
| Smart scraping pipeline | Fetch → Parse → Clean → Store |
| Auto-detect JS pages | Try requests first, fall back to Selenium |
| Multi-page data collector | Pagination + data deduplication |
| Save to SQLite | Real database storage |
| Project: Job listings | Title, company, salary → CSV |
| Project: Price tracker | Track price changes over time |

> Open `03_Pipelines_and_Real_World.ipynb` when ready!